# Signature verification - training and evaluation (CEDAR + BHSig260 + SigResize)

Trains the Stage 3 ConvNeXt-Tiny metric-learning encoder and evaluates it, on a Kaggle GPU.

**Prerequisite.** The attached dataset must have been preprocessed with the current Stage 2.2
path: pure-white background, true grayscale ink preserved, `256 x 512` canvas. Build it
locally with:

```bash
python -m preprocessing.build_manifests
python -m preprocessing.preprocess_dataset \\
  --manifest data/manifest_three.csv --image-root data \\
  --output-dir data/grayscale_processed --grayscale
```

then zip `data/grayscale_processed/` **flat** (`manifest.csv` at the archive root, image
folders beside it) and upload it as a private Kaggle dataset.

**What runs here**

1. environment + dataset checks
2. Stage 3 training **with augmentation** (`--augment`, training split only)
3. optional no-augmentation baseline for the ablation
4. fusion config fitted on the validation split
5. evaluation: deep vs fused, plus a per-dataset breakdown
6. artifact packaging


In [ ]:
# Cell 1 - clone the repo and install only what Kaggle is missing.
# Kaggle images already ship torch/torchvision; reinstalling them wastes ~2 GB and time.
# Change REPO_URL if you are training from a different fork.
REPO_URL = "https://github.com/acharyayush/Signature-Verification-FIRST-.git"
!git clone $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
# Augmentation is pure OpenCV/NumPy, so albumentations is not needed.
!pip install -q --no-input timm opencv-python scikit-learn


In [ ]:
# Cell 2 - verify the checkout and the GPU.
import sys, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print(sys.version)


In [ ]:
# Cell 3 - point at the attached preprocessed dataset.
# Use the notebook UI: Add Data -> Your Datasets -> attach the preprocessed dataset.
PROCESSED = "/kaggle/input/signature-verification-processed"
!ls -la {PROCESSED}
!head -3 {PROCESSED}/manifest.csv


In [ ]:
# Cell 4 - verify counts and that every relative image path resolves.
import csv, collections
from pathlib import Path

root = Path(PROCESSED)
rows = list(csv.DictReader((root / "manifest.csv").open()))
print("rows    ", len(rows))
print("splits  ", collections.Counter(r["split"] for r in rows))
print("signers ", len({r["signer_id"] for r in rows}))
print("datasets", collections.Counter(r["image_path"].split("/")[0] for r in rows))
missing = [r["image_path"] for r in rows if not (root / r["image_path"]).exists()]
print("missing images", len(missing), missing[:3])
assert not missing, "dataset incomplete or manifest paths do not match the layout"


## 1. Train (with augmentation)

`--augment` applies tilt (up to 20 deg for `standard`), shear, mild perspective, elastic warp,
pen stroke width, ink contrast, blur and noise (gamma is off: it duplicates ink contrast) - to the **training split only**.
Validation is never augmented, because the checkpoint is selected by the validation EER.
Each augmented image is re-whitened and re-fitted through the same crop + resize as
inference, so the representation contract is unchanged.

`--balance-datasets` gives every source dataset an equal share of each batch, so BHSig260
(160 + 100 signers) does not crowd out CEDAR (55). `--select-metric eer` picks the checkpoint
by the reported metric rather than by the triplet-loss scale.

Expect roughly 25 epochs on a T4. Best weights land in `checkpoints/stage3_aug/best_model.pt`.
If you hit CUDA OOM, drop `--signers-per-batch` to 8.


In [ ]:
# Cell 5 - Stage 3 training with augmentation (recommended run).
!python -m training.train_siamese \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --output-dir checkpoints/stage3_aug \
  --device cuda \
  --phase1-epochs 5 \
  --phase2-epochs 20 \
  --signers-per-batch 12 \
  --genuine-per-signer 4 \
  --forgeries-per-signer 2 \
  --augment \
  --augment-strength standard \
  --balance-datasets \
  --select-metric eer \
  --workers 2 \
  --seed 42


## 2. Optional baseline (no augmentation)

Same settings without `--augment`, so the two runs differ only by augmentation. Compare the
`eer_skilled` values from Cell 8 and the per-dataset numbers from Cell 9 to see whether the
augmentation actually helped on your data.


In [ ]:
# Cell 6 - baseline without augmentation (optional; skip to save GPU time).
!python -m training.train_siamese \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --output-dir checkpoints/stage3_base \
  --device cuda \
  --phase1-epochs 5 \
  --phase2-epochs 20 \
  --signers-per-batch 12 \
  --genuine-per-signer 4 \
  --forgeries-per-signer 2 \
  --balance-datasets \
  --select-metric eer \
  --workers 2 \
  --seed 42


## 3. Fit the score-level fusion config

Fitted on the **validation split only**, with signer-grouped folds, so the stored threshold is
not the optimistic in-sample one. `--fusion-type both` fits the weighted and logistic rules and
keeps the lower out-of-fold EER. `--normalize-hand dataset` z-scores the absolute handcrafted
features per dataset, which matters for a multi-script manifest.


In [ ]:
# Cell 7 - fit fusion on validation (uses the augmented checkpoint).
!python -m training.fit_fusion \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --fusion-type both \
  --cv-folds 5 \
  --fit-unit query \
  --normalize-hand dataset \
  --target-far 0.03 \
  --output checkpoints/fusion/fusion.json


## 4. Evaluate on the test split

Cell 8 prints the deep baseline block and the fusion block side by side. The threshold comes
from the validation split / fusion config; test signers are never used for tuning.


In [ ]:
# Cell 8 - deep vs fused, pooled over all test signers.
!python -m training.evaluate \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --target-far 0.03 \
  --fusion-config checkpoints/fusion/fusion.json \
  --fusion-report checkpoints/fusion/report.json


Cell 9 breaks the same numbers down per source dataset (`--group-depth 1` groups by
`BHSig260` / `CEDAR` / `SigResize`). This is where you see whether augmentation helped CEDAR
(historically the weakest) without damaging BHSig260's connected Indic strokes.


In [ ]:
# Cell 9 - per-dataset breakdown.
!python -m training.evaluate_per_dataset \
  --checkpoint checkpoints/stage3_aug/best_model.pt \
  --manifest {PROCESSED}/manifest.csv \
  --already-preprocessed \
  --references 5 \
  --device cuda \
  --batch-size 32 \
  --group-depth 1 \
  --target-far 0.03 \
  --fusion-config checkpoints/fusion/fusion.json \
  --report checkpoints/fusion/per_dataset.json


## 5. Optional: train on a subset of datasets

To train on CEDAR + BHSig260 only, write a filtered manifest and point Cells 5-9 at it. The
per-dataset augmentation presets are resolved from the path layout, so BHSig260 still gets the
milder elastic warp automatically.


In [ ]:
# Cell 10 - (optional) filter the manifest to chosen datasets.
import csv
from pathlib import Path

KEEP = {"CEDAR", "BHSig260"}
src = Path(PROCESSED) / "manifest.csv"
out = Path("/kaggle/working/manifest_subset.csv")
rows = [r for r in csv.DictReader(src.open()) if r["image_path"].split("/")[0] in KEEP]
with out.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["image_path", "signer_id", "is_forgery", "split"]
    )
    writer.writeheader()
    writer.writerows(rows)
print("kept", len(rows), "rows ->", out)


## 6. Package the artifacts and re-run the tests

`/kaggle/working` is what Kaggle saves as notebook output; `checkpoints/` is gitignored inside
the clone, so tarball what you want to keep.


In [ ]:
# Cell 11 - package artifacts and confirm the repo is intact.
%cd /kaggle/working/repo
!ls -lh checkpoints/stage3_aug/best_model.pt checkpoints/fusion/fusion.json
!tar -czf /kaggle/working/artifacts.tar.gz \
  checkpoints/stage3_aug/best_model.pt \
  checkpoints/fusion/fusion.json \
  checkpoints/fusion/report.json \
  checkpoints/fusion/per_dataset.json
!ls -lh /kaggle/working/artifacts.tar.gz
!python -m unittest discover -s tests -t . 2>&1 | tail -5
